## Data cleaning
In this step we cleen the data


we divide data into 3 section:
1) Spacial:
province, District

2) Temporal:
Date, Month, Time_of_day, year

3) Accident Condition:
Road_type, Intersection_type, Weather, Speed, Surface etc


## Reading Data

In [112]:
import pandas as pd

df=pd.read_csv('../Data/Nepal_Road_Traffic_Accidents_2023_2026.csv')

print(f"Loaded data : {df.shape[0]} rows and {df.shape[1]} columns")
df.head(3)



Loaded data : 1200 rows and 15 columns


,Accident_ID,Accident_Date,Year,Month,Province,District,Road_Type,Intersection_Type,Time_of_Day,Accident_Severity,Weather_Condition,Road_Surface,Vehicle_Type,Speed_Zone,Driver_Gender
0,1179,2026-01-09,2026,January,Sudurpashchim,Kailali,Highway,Junction,Afternoon,Damage Only,Clear,Slippery,Scooter,50,Male
1,866,2025-03-07,2025,March,Bagmati,Chitwan,Feeder,Curve,Evening,Serious Injury,Clear,Dry,Car,50,Female
2,102,2023-03-23,2023,March,Lumbini,Kapilvastu,Urban,Straight,Morning,Damage Only,Clear,Dry,Truck,30,Male


In [113]:
print("Shape:", df.shape)
print("Column Names: ")
for i , col in enumerate(df.columns):
    print(f"{i+1}. {col}")


Shape: (1200, 15)
Column Names: 
1. Accident_ID
2. Accident_Date
3. Year
4. Month
5. Province
6. District
7. Road_Type
8. Intersection_Type
9. Time_of_Day
10. Accident_Severity
11. Weather_Condition
12. Road_Surface
13. Vehicle_Type
14. Speed_Zone
15. Driver_Gender


In [114]:
df.dtypes


Accident_ID          int64
Accident_Date          str
Year                 int64
Month                  str
Province               str
District               str
Road_Type              str
Intersection_Type      str
Time_of_Day            str
Accident_Severity      str
Weather_Condition      str
Road_Surface           str
Vehicle_Type           str
Speed_Zone           int64
Driver_Gender          str
dtype: object

Problem1:Here date Time is in string format so we must change it in datatime Format

## Checking for Null Value(Missing Value)

In [115]:
Missing= df.isnull().sum() ## Calculate missing value and sums in each column
print("Missing Values in each column:")
print(Missing)
print(f"\n total missing values: {Missing.sum()}") ##add all the missing values in all the columns
print(f"Dataset completeness: {(1 - Missing.sum()/(df.shape[0]*df.shape[1]))*100:.1f}%")

Missing Values in each column:
Accident_ID          0
Accident_Date        0
Year                 0
Month                0
Province             0
District             0
Road_Type            0
Intersection_Type    0
Time_of_Day          0
Accident_Severity    0
Weather_Condition    0
Road_Surface         0
Vehicle_Type         0
Speed_Zone           0
Driver_Gender        0
dtype: int64

 total missing values: 0
Dataset completeness: 100.0%


## Checking for Duplicate Rows

In [116]:
full_dups=df.duplicated().sum()  #calc no of duplicated rows then add it to full_dups variable
print(f"Full duplicate rows: {full_dups}")

id_dups=df.duplicated(subset=['Accident_ID']).sum()
print(f"Duplicate IDs: {id_dups}")

print(f"Id Range: {df['Accident_ID'].min()} to {df['Accident_ID'].max()}")
print("Unique Ids:", df['Accident_ID'].nunique())

Full duplicate rows: 0
Duplicate IDs: 0
Id Range: 1 to 1200
Unique Ids: 1200


## Strip Leading/ Trailing Whitespaces

In [117]:
str_cols=df.select_dtypes(include='str') 
print("String columns:")
print(list(str_cols.columns)) #select only columns


for col in str_cols.columns:
    df[col]=df[col].str.strip() #strip leading and trailing spaces from string columns

print("Whitespace removed from string columns.")

String columns:
['Accident_Date', 'Month', 'Province', 'District', 'Road_Type', 'Intersection_Type', 'Time_of_Day', 'Accident_Severity', 'Weather_Condition', 'Road_Surface', 'Vehicle_Type', 'Driver_Gender']
Whitespace removed from string columns.


## Convert Accident_Date to datetime

In [118]:
df['Accident_Date']=pd.to_datetime(df['Accident_Date'])
print("NewDataType of Accident_Date:", df['Accident_Date'].dtype)
print("earliest date:", df['Accident_Date'].min())
print("Latest date:", df['Accident_Date'].max())


month_mismatch=(df['Month']!=df['Accident_Date'].dt.month_name())
year_mismatch=df['Year']!=df['Accident_Date'].dt.year


print(f"Month mismatch count: {month_mismatch.sum()}")
print(f"Year mismatch count: {year_mismatch.sum()}")


NewDataType of Accident_Date: datetime64[us]
earliest date: 2023-01-02 00:00:00
Latest date: 2026-01-24 00:00:00
Month mismatch count: 0
Year mismatch count: 0


## Validating Categorical Value

In [119]:
expected = {
    'Road_Type':         ['Highway', 'Urban', 'Rural', 'Feeder'],
    'Intersection_Type': ['Straight', 'Junction', 'Curve', 'Roundabout'],
    'Time_of_Day':       ['Morning', 'Afternoon', 'Evening', 'Night'],
    'Accident_Severity': ['Fatal', 'Serious Injury', 'Minor Injury', 'Damage Only'],
    'Weather_Condition': ['Clear', 'Rain', 'Fog', 'Dusty'],
    'Road_Surface':      ['Dry', 'Wet', 'Slippery', 'Damaged'],
    'Vehicle_Type':      ['Motorbike','Car','Bus','Truck','Scooter','Jeep','Microbus'],
    'Driver_Gender':     ['Male', 'Female', 'Other'],
    'Speed_Zone':        [30, 50, 80],
}

for col, vals in expected.items():
    actual = set(df[col].unique())
    unexpected = actual - set(vals)
    status = "OK" if not unexpected else f" Unexpected: {unexpected}"
    print(f"  {col:<30} {status}")

  Road_Type                      OK
  Intersection_Type              OK
  Time_of_Day                    OK
  Accident_Severity              OK
  Weather_Condition              OK
  Road_Surface                   OK
  Vehicle_Type                   OK
  Driver_Gender                  OK
  Speed_Zone                     OK


## Sort By Date

In [120]:
# Sort by date ascending (earliest first), reset index
df = df.sort_values(['Accident_Date', 'Accident_ID'], ascending=[True, True]).reset_index(drop=True)

print("First 3 rows after sorting:")
print(df[['Accident_Date', 'Province', 'Accident_Severity']].head(3))

print("\nLast 3 rows after sorting:")
print(df[['Accident_Date', 'Province', 'Accident_Severity']].tail(3))

First 3 rows after sorting:
  Accident_Date Province Accident_Severity
0    2023-01-02  Karnali    Serious Injury
1    2023-01-03  Madhesh       Damage Only
2    2023-01-06  Bagmati      Minor Injury

Last 3 rows after sorting:
     Accident_Date       Province Accident_Severity
1197    2026-01-22  Sudurpashchim             Fatal
1198    2026-01-23        Bagmati    Serious Injury
1199    2026-01-24        Lumbini      Minor Injury


Shifting Index By one

In [121]:
# Shift index: add 1 so it starts at 1 instead of 0
df.index = df.index + 1
df.index.name = 'Index'

print("Index range:", df.index[0], "→", df.index[-1])
print("Index name: ", df.index.name)
print()
df.head(4)

Index range: 1 → 1200
Index name:  Index



,Accident_ID,Accident_Date,Year,Month,Province,District,Road_Type,Intersection_Type,Time_of_Day,Accident_Severity,Weather_Condition,Road_Surface,Vehicle_Type,Speed_Zone,Driver_Gender
Index,,,,,,,,,,,,,,,
1,1,2023-01-02,2023,January,Karnali,Dailekh,Highway,Straight,Afternoon,Serious Injury,Clear,Wet,Scooter,80,Female
2,2,2023-01-03,2023,January,Madhesh,Sarlahi,Highway,Junction,Afternoon,Damage Only,Clear,Dry,Motorbike,30,Male
3,3,2023-01-06,2023,January,Bagmati,Bhaktapur,Rural,Junction,Evening,Minor Injury,Clear,Slippery,Truck,50,Male
4,4,2023-01-06,2023,January,Bagmati,Lalitpur,Feeder,Junction,Evening,Minor Injury,Clear,Slippery,Motorbike,80,Male


## Save to cleaned Dataset

In [122]:

Output_path='../Data/Nepal_Road_Traffic_Accidents_Cleaned.csv'
df.to_csv(Output_path, index=False)

print(f"Saved:{Output_path} with {df.shape[0]} rows and {df.shape[1]} columns")

Saved:../Data/Nepal_Road_Traffic_Accidents_Cleaned.csv with 1200 rows and 15 columns
